# تدريب صوت خاص من تسجيلات حقيقية

النتيجة: نموذج **Piper** صغير بصوت الشخص نفسه وبلهجته، يتعلّم من تسجيلاته في صفحة التسجيل (المستودع الخاص
`omran77/omran-voices-recordings`)، وينحفظ في المستودع الخاص `omran77/omran-voices-piper`. كل شي مجاني.

**على Kaggle:** يمين الصفحة ← Settings ← Accelerator **GPU T4 x2** و Internet **On**. فوق ← Add-ons ← Secrets ←
علّم على `HF_TOKEN` (مفتاح Hugging Face من نوع **Write**، نفس اللي استخدمته المرة اللي فاتت). بعدين فوق يمين
**Save Version** ← **Save & Run All (Commit)**. يشتغل بروحه حتى لو سكّرت الصفحة.

**على Google Colab:** Runtime ← Change runtime type ← **T4 GPU**، وأيقونة المفتاح 🔑 يسار ← `HF_TOKEN` مع
**Notebook access**، ثم Runtime ← **Run all** والصفحة مفتوحة.

التشغيلة تاخذ تقريبًا ١٠ ساعات (التدريب ٩). لما يخلص: انزل لآخر الصفحة وتلقى العيّنات تسمعها على طول. تشغيلة
ثانية تكمل التدريب من آخر نسخة، وتاخذ أي تسجيلات جديدة انضافت.

In [ ]:
# ===== الإعدادات =====
SPEAKER = 'سيف'                        # الاسم مثل ما انكتب في صفحة التسجيل
VOICE = 'saif'                         # اسم الصوت في الملفات (حروف إنجليزي صغيرة)
REC_REPO = 'omran77/omran-voices-recordings'
MODEL_REPO = 'omran77/omran-voices-piper'
SENTENCE_URLS = ['https://omran-voices.vercel.app/data/sentences.json', 'https://omran-voices.vercel.app/data/dialects.json']
WHISPER = 'large-v3'                   # يفرّغ التسجيلات (التسجيل المتواصل ما له نص)
TRAIN_HOURS = 9.0                      # وقت التدريب في هالتشغيلة (كرت T4 يمشي ~٠٫٧ خطوة بالثانية: ٩ ساعات ≈ ٢٢ ألف خطوة)؛ تشغيلة ثانية تكمل
ROUND = 2                              # يزيد لما تتغيّر طريقة تجهيز الصوت: التدريب يبدأ من جديد وما يكمل على نسخة تعلّمت من صوت مكتوم
SESSION_HOURS = 11.5                   # أطول جلسة (Kaggle ١٢ ساعة)
UPLOAD_EVERY_MIN = 60                  # نسخة وعيّنات كل ساعة (كل نسخة ~٩٠٠ ميجا في المستودع الخاص)
VOICE_FILE = 'ar_AE-{}_gulf-medium'    # ar_AE-saif_gulf-medium.onnx

In [ ]:
import json, os, shutil, subprocess, tarfile, time, urllib.request
from pathlib import Path

T0 = time.time()
# Kaggle keeps /kaggle/working after the run and caps it at 20 GB: the work happens outside it, and
# everything worth keeping goes to the Hugging Face repo.
W = Path('/tmp/omran') if Path('/kaggle/working').exists() else next(p for p in (Path('/content'), Path.cwd()) if p.exists())
W.mkdir(parents=True, exist_ok=True)
os.chdir(W)
print('working in', W, '-', shutil.disk_usage(W).free // 2**30, 'GB free')


def run(cmd, log=None, env=None):
    """Runs a command and shows its output here (the notebook hides a child process's own output)."""
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if log is None:
            print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'failed ({p.returncode}): {cmd}\n' + ''.join(tail[-15:]))


def out(cmd, env=None):
    return subprocess.run(cmd, capture_output=True, text=True, check=True, env=env).stdout.strip()


def hours_left():
    return SESSION_HOURS - (time.time() - T0) / 3600


def secret(name):
    """The secret from Kaggle (Add-ons > Secrets) or Colab (the key icon), whichever this is."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except ImportError:
        pass
    try:
        from google.colab import userdata
    except ImportError:
        return os.environ.get(name, '')
    try:
        return userdata.get(name)
    except Exception as e:
        raise RuntimeError('ضيف السر HF_TOKEN من أيقونة المفتاح يسار الصفحة، وفعّل Notebook access') from e


HF_TOKEN = (secret('HF_TOKEN') or '').strip()
assert HF_TOKEN.startswith('hf_'), 'السر HF_TOKEN ناقص أو مب مفتاح Hugging Face'
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
api = HfApi(token=HF_TOKEN)
print('Hugging Face account:', api.whoami()['name'])

gpus = [g for g in subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                                  capture_output=True, text=True).stdout.splitlines() if g.strip()]
print('GPUs:', gpus)
assert gpus, 'ما في كرت: Kaggle ← Settings ← Accelerator ← GPU T4 x2 (أو Colab ← Runtime ← Change runtime type ← T4 GPU)'

api.create_repo(MODEL_REPO, private=True, exist_ok=True)
assert api.model_info(MODEL_REPO).private, MODEL_REPO + ' لازم يكون Private'
assert api.dataset_info(REC_REPO).private, REC_REPO + ' لازم يكون Private'
REPO_FILES = set(api.list_repo_files(MODEL_REPO))

In [ ]:
%%writefile prep_real.py
"""One person's real recordings -> Piper training data (the owner, 21 Nov: «خاص لي انا من البنت والرجل»).

  python prep_real.py transcribe --rec rec --sentences sentences.json --out words.json
  python prep_real.py cut --rec rec --words words.json --sentences sentences.json --speaker saif --out data
  python prep_real.py check --data data --speaker saif
  python prep_real.py list --data data --speaker saif --out data/saif.csv

rec/ holds one speaker's folder of the private recordings dataset: b…/…wav with metadata.jsonl
(api/_recordings.js). Two kinds of recording are in it:
  - one sentence, its text known (the «جملة جملة» way);
  - a piece of a continuous take («تسجيل متواصل», free: true): the person read down a list or just
    talked, so there is no text, only the sentences that were on screen (prompt_ids).

transcribe: Whisper writes every recording as words with their times.
cut: the words become utterances of 1.5–12 s, cut at the pauses; an utterance that is one of the
  sentences on screen takes that sentence's spelling (Whisper misspells dialect words); a clip whose
  length does not fit its text, or that Whisper was unsure of, is left out. Clips are 22 050 Hz.
  A muffled recording (covered or far microphone) is given the balance of clear speech back.
check: each clip is heard once more on its own; a clip whose text the second listening does not
  agree with is left out.
list: Piper's training list, each text with the diacritics and the pausal endings the site will
  give this voice when it speaks (the same code: piper_fn.py is the site's api/piper.py).

Gulf speech stops on each word (اليوم، مب اليومُ): the diacritizer adds formal case endings that the
recordings do not have, so the last letter of each word gets a sukun (pausal), in training and later
in the site alike. Then the model hears what its text says.
"""
import argparse
import difflib
import json
import re
import sys
import wave
from pathlib import Path

SR_OUT = 22050
MIN_SEC, MAX_SEC = 1.5, 12.0
GAP_SPLIT = 0.35        # a pause this long ends an utterance, once the utterance is long enough
GAP_ALWAYS = 0.8        # a pause this long always ends it
MIN_SPLIT_SEC = 1.5     # a five-word sentence from the list takes about 2.5 s: its pause must end it
PAD_BEFORE, PAD_AFTER = 0.12, 0.18
MATCH = 0.78            # how alike an utterance and a sentence on screen must be to take its spelling
MIN_PROB = 0.5          # Whisper's mean word confidence below this: the text is a guess
MIN_SPOT_PROB = 0.3     # the same, for an utterance that matches a sentence on screen (the match is evidence too)
AGREE = 0.7             # a clip heard again on its own must share this much of its letters with its text
MIN_KEPT = 0.5          # a second listening that rejects more than half of the clips is not trusted
WHISPER_HINT = 'هلا والله، شحالك؟ وايد زين. بغيت أروح الحين. شو تبا؟ جذي أحسن.'

AR = 'ءآأؤإئابةتثجحخدذرزسشصضطظعغفقكلمنهوىي'
MARKS = 'ًٌٍَُِّْٰـ'
WORD = re.compile('[' + AR + MARKS + ']+')
SUKUN, FATHA, FATHATAN = 'ْ', 'َ', 'ً'


# --- text ---------------------------------------------------------------------------------------
def bare(text):
    """The letters only, in one spelling: what two writings of the same words share."""
    t = ''.join(ch for ch in str(text) if ch not in MARKS)
    t = re.sub('[إأآ]', 'ا', t).replace('ة', 'ه').replace('ى', 'ي')
    return ' '.join(re.sub('[^' + AR + ' ]', ' ', t).split())


def letters(text):
    return sum(1 for ch in text if ch in AR)


def accept(seconds, text):
    """Spoken Arabic runs about 0.06–0.09 s per letter; real talk has pauses, so the slow side is
    wider than for a generated clip. Far outside: the clip is not this text."""
    n = letters(text)
    if n < 6 or not MIN_SEC <= seconds <= MAX_SEC:
        return False
    return 0.035 <= seconds / n <= 0.22


def best_prompt(text, candidates):
    """The sentence on screen that this utterance is, or None."""
    a = bare(text).replace(' ', '')
    best, score = None, 0.0
    for c in candidates:
        b = bare(c).replace(' ', '')
        if not b or not 0.75 <= len(a) / len(b) <= 1.33:
            continue
        r = difflib.SequenceMatcher(None, a, b).ratio()
        if r > score:
            best, score = c, r
    return best if score >= MATCH else None


def pausal_word(word):
    idx = [i for i, c in enumerate(word) if c in AR]
    if len(idx) < 2:  # و، ب، ل alone: nothing to stop on
        return word
    last = idx[-1]
    head, tail = word[:last], word[last + 1:]
    if word[last] in 'اى':  # long -aa, and -an written ًا or اً read -aa: fatha before the alif
        head, tail = head.replace(FATHATAN, ''), tail.replace(FATHATAN, '')
        before = head[len(head.rstrip(MARKS)):]
        if not any(c in before for c in 'َُِ'):
            head += FATHA
        return head + word[last] + tail
    # a final shadda + sukun is still read doubled with a vowel by Piper's phonemizer, and «ةْ» is read «-at»
    return head + ('ه' if word[last] == 'ة' else word[last]) + SUKUN


def pausal(text):
    """Every Arabic word ends on a sukun: الْيَوْمُ -> الْيَوْمْ، قَهْوَةً -> قَهْوَهْ، شُكْرًا -> شُكْرَا."""
    return WORD.sub(lambda m: pausal_word(m.group(0)), text)


# --- the recordings -----------------------------------------------------------------------------
def read_meta(rec_dir):
    """Every consented recording under rec_dir, in the order it was made."""
    rows = []
    for meta in sorted(Path(rec_dir).rglob('metadata.jsonl')):
        for line in meta.read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            m = json.loads(line)
            wav = meta.parent / m.get('file_name', '')
            if m.get('consent') is not True or not wav.is_file():
                continue
            rows.append({
                'key': wav.relative_to(rec_dir).as_posix(), 'path': wav, 'text': m.get('text') or '',
                'free': m.get('free') is True, 'session': m.get('session') or '', 'part': int(m.get('part') or 0),
                'prompt_ids': [i for i in (m.get('prompt_ids') or []) if isinstance(i, int)],
                'duration': float(m.get('duration') or 0), 'at': m.get('recorded_at') or '',
            })
    rows.sort(key=lambda r: (r['at'], r['session'], r['part'], r['key']))
    return rows


def utterances(words):
    """Words with times -> runs of words that are said in one breath."""
    out, cur = [], []

    def flush():
        if cur:
            out.append(list(cur))
            cur.clear()

    for w in words:
        if cur:
            gap = w['s'] - cur[-1]['e']
            length = cur[-1]['e'] - cur[0]['s']
            ended = cur[-1]['w'][-1:] in '.!?؟'
            if gap >= GAP_ALWAYS or (length >= MIN_SPLIT_SEC and (gap >= GAP_SPLIT or ended)):
                flush()
            elif w['e'] - cur[0]['s'] > MAX_SEC:
                # too long without a real pause: cut at the longest gap there is
                gaps = [cur[i + 1]['s'] - cur[i]['e'] for i in range(len(cur) - 1)]
                if gaps:
                    k = max(range(len(gaps)), key=gaps.__getitem__) + 1
                    rest = cur[k:]
                    del cur[k:]
                    flush()
                    cur.extend(rest)
                else:
                    flush()
        cur.append(w)
    flush()
    return out


def spot(words, candidates):
    """The sentences on screen, found in the words wherever they were read: [(first word, after last, sentence)].
    A person reading down the list may not pause between two sentences, so pauses alone cannot cut them."""
    hits = []
    for c in dict.fromkeys(candidates):
        target = bare(c).replace(' ', '')
        n = len(bare(c).split())
        if len(target) < 6:
            continue
        best = None
        for i in range(len(words)):
            for j in range(i + max(2, n - 2), min(len(words), i + n + 2) + 1):
                got = bare(' '.join(w['w'] for w in words[i:j])).replace(' ', '')
                if not got or not 0.75 <= len(got) / len(target) <= 1.33:
                    continue
                r = difflib.SequenceMatcher(None, got, target).ratio()
                if best is None or r > best[0]:
                    best = (r, i, j, c)
        if best and best[0] >= MATCH:
            hits.append(best)
    taken, out = set(), []
    for r, i, j, c in sorted(hits, key=lambda h: -h[0]):  # the surest first; a word belongs to one sentence
        if taken.isdisjoint(range(i, j)):
            taken.update(range(i, j))
            out.append((i, j, c))
    return sorted(out)


def plan(row, words, sentence_of):
    """One recording -> the clips to cut from it: [(start, end, text)], and why the rest was dropped."""
    drops = {}

    def drop(why):
        drops[why] = drops.get(why, 0) + 1

    if not row['free'] and row['text'] and accept(row['duration'], row['text']):
        return [(0.0, row['duration'], row['text'])], drops  # one sentence, said as written
    cands = [sentence_of[i] for i in row['prompt_ids'] if i in sentence_of]
    if not row['free'] and row['text']:
        cands.append(row['text'])  # a «جملة جملة» take that ran long: the sentence may still be in it
    clips, at = [], 0
    spans = spot(words, cands)
    for i, j, sentence in spans + [(len(words), len(words), None)]:
        for u in utterances(words[at:i]):  # what was said between two sentences from the screen: his own words
            start, end = u[0]['s'], u[-1]['e']
            text = ' '.join(w['w'] for w in u).strip()
            known = best_prompt(text, cands)
            if known:
                text = known
            elif sum(w['p'] for w in u) / len(u) < MIN_PROB:
                drop('unsure')
                continue
            if not accept(end - start, text):
                drop('length')
                continue
            clips.append((start, end, text))
        if sentence:
            start, end = words[i]['s'], words[j - 1]['e']
            # the sentences on screen are also Whisper's hint: one it wrote without hearing it has low confidence
            if sum(w['p'] for w in words[i:j]) / (j - i) < MIN_SPOT_PROB:
                drop('unsure')
            elif accept(end - start, sentence):
                clips.append((start, end, sentence))
            else:
                drop('length')
        at = j
    return clips, drops


# --- audio ---------------------------------------------------------------------------------------
def read_wav(path):
    import numpy as np
    with wave.open(str(path)) as w:
        assert w.getnchannels() == 1 and w.getsampwidth() == 2, 'mono 16-bit expected: ' + str(path)
        return np.frombuffer(w.readframes(w.getnframes()), dtype='<i2').astype('float32') / 32768.0, w.getframerate()


def resample(y, sr, to):
    if sr == to:
        return y
    from math import gcd
    from scipy.signal import resample_poly
    g = gcd(sr, to)
    return resample_poly(y, to // g, sr // g).astype('float32')


def write_clip(path, y, sr):
    import numpy as np
    y = resample(y, sr, SR_OUT)
    peak = float(np.max(np.abs(y))) if y.size else 0.0
    if peak > 0:
        y = y * (0.89 / peak)  # -1 dBFS, every clip alike
    with wave.open(str(path), 'wb') as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(SR_OUT)
        w.writeframes((np.clip(y, -1, 1) * 32767).astype('<i2').tobytes())


# --- a muffled recording ---------------------------------------------------------------------------
# The first voice trained here (21 Nov) came out «صوت هواء ومش واضح». The model was not at fault: every
# one of the 280 clips was muffled at the source (a covered or far microphone). 2-4 kHz, where the
# consonants are, sat 16 dB under clear speech, and the model copied what it heard. The consonants
# were still in the recording (20 dB over the room's noise there), only far too quiet. So before
# training: the speaker's long-term spectrum is compared with clear speech; when it is muffled, the
# room's noise is turned down first (or the next step would turn it into hiss) and one fixed
# equalizer for the whole speaker brings the balance back. A clear recording is left untouched.
EQ_STEP = 250           # Hz
# Clear speech (three studio male voices and the voice the training starts from), louder half of the
# frames, dB per 250 Hz, 0 = the mean of 250-1000 Hz.
CLEAR_SPEECH_DB = [2.0, 1.9, 0.0, -4.2, -8.2, -9.7, -11.1, -12.0, -13.0, -13.6, -13.7, -14.4, -15.2, -17.7, -22.1, -25.9,
                   -30.1, -32.4, -34.4, -34.5, -34.3, -33.8, -33.1, -32.2, -30.9, -30.3, -30.5, -30.8, -30.7, -30.6, -30.3,
                   -30.9, -30.7, -30.1, -29.8, -30.5, -31.2, -32.0, -32.6, -33.0, -33.3, -33.2, -34.3, -34.6]
MUFFLED_FROM, MUFFLED_TO = 1500, 4000
MUFFLED_DB = 6.0        # that band this far under clear speech: the recording is muffled
EQ_MAX, EQ_MAX_HIGH, EQ_HIGH_FROM, EQ_MIN = 20.0, 12.0, 5000, -6.0  # above 5 kHz there is little but noise to raise
GATE_FLOOR_DB, GATE_OVER = -15.0, 2.0
N_FFT, HOP = 1024, 256


def _stft(y):
    import numpy as np
    y = np.pad(np.asarray(y, dtype='float64'), N_FFT // 2, mode='reflect')
    n = 1 + (len(y) - N_FFT) // HOP
    idx = np.arange(N_FFT)[None, :] + HOP * np.arange(n)[:, None]
    return np.fft.rfft(y[idx] * np.hanning(N_FFT + 1)[:-1], axis=1)


def _istft(X, length):
    import numpy as np
    win = np.hanning(N_FFT + 1)[:-1]
    frames = np.fft.irfft(X, n=N_FFT, axis=1) * win
    out = np.zeros(HOP * (len(X) - 1) + N_FFT)
    norm = np.zeros_like(out)
    for i, f in enumerate(frames):
        out[i * HOP:i * HOP + N_FFT] += f
        norm[i * HOP:i * HOP + N_FFT] += win ** 2
    out = out / np.maximum(norm, 1e-6)
    return out[N_FFT // 2:N_FFT // 2 + length].astype('float32')


def speech_spectrum(clips):
    """The speaker's long-term spectrum from clips at SR_OUT, in CLEAR_SPEECH_DB's terms."""
    import numpy as np
    edges = np.arange(len(CLEAR_SPEECH_DB) + 1) * EQ_STEP
    f = np.fft.rfftfreq(N_FFT, 1 / SR_OUT)
    acc = np.zeros(len(CLEAR_SPEECH_DB))
    for y in clips:
        if len(y) < N_FFT:
            continue
        P = np.abs(_stft(y)) ** 2
        e = P.sum(1)
        loud = P[e >= np.median(e)].mean(0)  # the frames where there is speech
        band = np.array([loud[(f >= a) & (f < b)].sum() for a, b in zip(edges, edges[1:])])
        if band.sum() > 0:
            acc += band / band.sum()
    assert acc.sum() > 0, 'no audio to measure'
    d = 10 * np.log10(acc / acc.sum() + 1e-12)
    return d - d[1:4].mean()


def muffled_by(speaker_db):
    """dB that 1.5-4 kHz is under clear speech (negative: brighter than it)."""
    import numpy as np
    a, b = MUFFLED_FROM // EQ_STEP, MUFFLED_TO // EQ_STEP
    return float(np.mean(np.asarray(CLEAR_SPEECH_DB[a:b]) - np.asarray(speaker_db[a:b])))


def eq_curve(speaker_db):
    """The equalizer, dB per 250 Hz: clear speech minus the speaker, smoothed and capped."""
    import numpy as np
    g = np.asarray(CLEAR_SPEECH_DB) - np.asarray(speaker_db)
    g = np.convolve(np.pad(g, 2, mode='edge'), np.ones(5) / 5, mode='valid')
    hz = np.arange(len(g)) * EQ_STEP
    g = np.clip(g, EQ_MIN, np.where(hz >= EQ_HIGH_FROM, EQ_MAX_HIGH, EQ_MAX))
    g[0] = min(g[0], 0.0)  # never raise rumble
    return g


def repair(y, eq_db):
    """One clip at SR_OUT: the room's noise down, then the equalizer."""
    import numpy as np
    X = _stft(y)
    power = np.abs(X) ** 2
    quiet = np.argsort(power.sum(1))[:max(3, len(power) // 10)]  # the clip's quietest frames: the room
    noise = power[quiet].mean(0)
    gate = np.sqrt(np.maximum(1 - GATE_OVER * noise[None, :] / np.maximum(power, 1e-12), 10 ** (GATE_FLOOR_DB / 10)))
    k = np.ones(5) / 5  # smoothed over time and frequency, or the gate chirps
    for axis in (0, 1):
        gate = np.apply_along_axis(lambda r: np.convolve(np.pad(r, 2, mode='edge'), k, mode='valid'), axis, gate)
    f = np.fft.rfftfreq(N_FFT, 1 / SR_OUT)
    eq = 10 ** (np.interp(f, np.arange(len(eq_db)) * EQ_STEP + EQ_STEP / 2, eq_db) / 20)
    return _istft(X * gate * eq[None, :], len(y))


def repair_clips(wav_dir):
    """Measures the clips in wav_dir and, when they are muffled, repairs them in place."""
    paths = sorted(Path(wav_dir).glob('*.wav'))
    before = speech_spectrum(read_wav(p)[0] for p in paths)
    sound = {'muffled_db': round(muffled_by(before), 1), 'repaired': False}
    if sound['muffled_db'] < MUFFLED_DB:
        return sound
    curve = eq_curve(before)
    for p in paths:
        y, sr = read_wav(p)
        write_clip(p, repair(y, curve), sr)
    sound.update(repaired=True, after_db=round(muffled_by(speech_spectrum(read_wav(p)[0] for p in paths)), 1),
                 eq_db=[round(float(v), 1) for v in curve])
    return sound


# --- commands ------------------------------------------------------------------------------------
def load_sentences(path):
    return {int(s[0]): s[1] for s in json.loads(Path(path).read_text(encoding='utf-8'))}


def cmd_transcribe(a):
    import whisper
    rows = read_meta(a.rec)
    out = Path(a.out)
    done = json.loads(out.read_text(encoding='utf-8')) if out.exists() else {}
    sentence_of = load_sentences(a.sentences)
    model = whisper.load_model(a.model)
    for n, row in enumerate(rows, 1):
        if row['key'] in done:
            continue
        y, sr = read_wav(row['path'])
        # the sentences that were on screen bias Whisper towards their spelling (it takes ~220 tokens)
        hint = (WHISPER_HINT + ' ' + ' '.join(sentence_of[i] for i in row['prompt_ids'] if i in sentence_of))[:420]
        r = model.transcribe(resample(y, sr, 16000), language='ar', word_timestamps=True, condition_on_previous_text=False,
                             temperature=0.0, initial_prompt=hint)
        done[row['key']] = [{'w': w['word'].strip(), 's': round(float(w['start']), 3), 'e': round(float(w['end']), 3),
                             'p': round(float(w.get('probability', 1.0)), 3)}
                            for seg in r['segments'] for w in seg.get('words', []) if w['word'].strip()]
        if n % 5 == 0 or n == len(rows):
            out.write_text(json.dumps(done, ensure_ascii=False), encoding='utf-8')
            print(f'{n}/{len(rows)} transcribed', flush=True)
    out.write_text(json.dumps(done, ensure_ascii=False), encoding='utf-8')
    return 0 if done else 1


def cmd_cut(a):
    rows = read_meta(a.rec)
    words = json.loads(Path(a.words).read_text(encoding='utf-8'))
    sentence_of = load_sentences(a.sentences)
    out = Path(a.out)
    (out / 'wav').mkdir(parents=True, exist_ok=True)
    lines, report, n = [], {'recordings': len(rows), 'clips': 0, 'seconds': 0.0, 'matched': 0, 'dropped': {}, 'examples': []}, 0
    for row in rows:
        clips, drops = plan(row, words.get(row['key'], []), sentence_of)
        for why, k in drops.items():
            report['dropped'][why] = report['dropped'].get(why, 0) + k
        if not clips:
            continue
        y, sr = read_wav(row['path'])
        known = {sentence_of[i] for i in row['prompt_ids'] if i in sentence_of} | {row['text']}
        for start, end, text in clips:
            a0, a1 = max(0, int((start - PAD_BEFORE) * sr)), min(len(y), int((end + PAD_AFTER) * sr))
            if a1 - a0 < MIN_SEC * sr * 0.8:
                continue
            n += 1
            name = f'{a.speaker}_{n:04d}.wav'
            write_clip(out / 'wav' / name, y[a0:a1], sr)
            lines.append(name + '\t' + text.replace('\t', ' ').replace('\n', ' '))
            report['clips'] += 1
            report['seconds'] += (a1 - a0) / sr
            report['matched'] += text in known
            if len(report['examples']) < 12:
                report['examples'].append({'file': name, 'seconds': round((a1 - a0) / sr, 1), 'text': text})
    report['seconds'] = round(report['seconds'], 1)
    if lines:
        report['sound'] = repair_clips(out / 'wav')
    (out / f'{a.speaker}.tsv').write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')
    (out / f'{a.speaker}.report.json').write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding='utf-8')
    print(json.dumps({k: v for k, v in report.items() if k != 'examples'}, ensure_ascii=False))
    for e in report['examples']:
        print(' ', e['file'], e['seconds'], 's', e['text'])
    return 0 if lines else 1


def agreement(text, heard):
    """How alike a clip's text and a second listening of the clip are: the letters, 0-1."""
    a, b = bare(text).replace(' ', ''), bare(heard).replace(' ', '')
    return difflib.SequenceMatcher(None, a, b).ratio() if a and b else 0.0


def keep_agreed(lines, heard_of):
    """The tsv lines whose text a second listening agrees with, and the names of the others."""
    kept, out = [], []
    for line in lines:
        name, text = line.split('\t', 1)
        if agreement(text, heard_of.get(name, '')) >= AGREE:
            kept.append(line)
        else:
            out.append(name)
    return kept, out


def cmd_check(a):
    """When the person talked instead of reading the list, every text is Whisper's guess from the whole
    recording. Each clip is heard once more on its own; where the two listenings differ a lot, the text
    is wrong or the cut took half a word, and the clip would teach the voice to mumble. It goes."""
    import whisper
    data = Path(a.data)
    tsv, rep = data / f'{a.speaker}.tsv', data / f'{a.speaker}.report.json'
    lines = [l for l in tsv.read_text(encoding='utf-8').splitlines() if l.strip()]
    model = whisper.load_model(a.model)
    heard = {}
    for n, line in enumerate(lines, 1):
        name = line.split('\t', 1)[0]
        y, sr = read_wav(data / 'wav' / name)
        heard[name] = model.transcribe(resample(y, sr, 16000), language='ar', condition_on_previous_text=False,
                                       temperature=0.0)['text']
        if n % 50 == 0 or n == len(lines):
            print(f'{n}/{len(lines)} heard again', flush=True)
    kept, out = keep_agreed(lines, heard)
    report = json.loads(rep.read_text(encoding='utf-8'))
    if len(kept) < len(lines) * MIN_KEPT:
        # a second listening that rejects most of the clips is itself wrong (no GPU, another language...)
        print(f'the second listening disagreed with {len(out)} of {len(lines)} clips: not trusted, nothing dropped')
        report['misheard'] = 'check not trusted'
    else:
        for name in out:
            (data / 'wav' / name).unlink()
        seconds = 0.0
        for line in kept:
            y, sr = read_wav(data / 'wav' / line.split('\t', 1)[0])
            seconds += len(y) / sr
        tsv.write_text('\n'.join(kept) + '\n', encoding='utf-8')
        report.update(clips=len(kept), seconds=round(seconds, 1), misheard=len(out))
        print(f'{len(out)} clips dropped (their text and the second listening differ), {len(kept)} kept, {seconds / 60:.1f} minutes')
    rep.write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding='utf-8')
    return 0


def build_list(data_dir, speaker, vowel):
    out, seen = [], set()
    for line in (Path(data_dir) / f'{speaker}.tsv').read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        name, text = line.split('\t', 1)
        if name in seen or not (Path(data_dir) / 'wav' / name).exists():
            continue
        seen.add(name)
        text = vowel(text).replace('|', ' ').strip()
        if text:
            out.append(f'{name}|{text}')
    return out


def site_vowel():
    """The text exactly as the site will give it to this voice: CATT's diacritics, then the pausal endings."""
    import piper_fn  # the site's api/piper.py, written next to this file by the notebook
    catt = piper_fn.catt()
    assert catt, 'CATT (the diacritizer) could not be loaded'
    return lambda t: pausal(catt(t))


def cmd_score(a):
    """How much of what the voice was asked to say Whisper hears in its samples: 0-1 (the voice the
    training starts from scores about 0.95; the first voice we trained, heard as «مش واضح», 0.7).
    Training is nine hours with nobody listening: this number, once an hour, says where it is."""
    import whisper
    import samples_real
    model = whisper.load_model(a.model)
    scores = []
    for i, text in enumerate(samples_real.TEXTS, 1):
        y, sr = read_wav(Path(a.samples) / f'{a.name}_{i}.wav')
        heard = model.transcribe(resample(y, sr, 16000), language='ar', condition_on_previous_text=False, temperature=0.0)['text']
        scores.append(agreement(text, heard))
    print('SCORE', round(sum(scores) / len(scores), 3))
    return 0


def cmd_list(a):
    lines = build_list(a.data, a.speaker, site_vowel())
    Path(a.out).write_text('\n'.join(lines) + '\n', encoding='utf-8')
    print(f'{a.speaker}: {len(lines)} clips, e.g. {lines[0] if lines else ""}')
    return 0 if lines else 1


def main(argv=None):
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest='cmd', required=True)
    t = sub.add_parser('transcribe')
    t.add_argument('--rec', required=True)
    t.add_argument('--sentences', required=True)
    t.add_argument('--out', required=True)
    t.add_argument('--model', default='large-v3')
    c = sub.add_parser('cut')
    c.add_argument('--rec', required=True)
    c.add_argument('--words', required=True)
    c.add_argument('--sentences', required=True)
    c.add_argument('--speaker', required=True)
    c.add_argument('--out', required=True)
    ch = sub.add_parser('check')
    ch.add_argument('--data', required=True)
    ch.add_argument('--speaker', required=True)
    ch.add_argument('--model', default='large-v3')
    sc = sub.add_parser('score')
    sc.add_argument('--samples', required=True)
    sc.add_argument('--name', required=True)
    sc.add_argument('--model', default='large-v3')
    li = sub.add_parser('list')
    li.add_argument('--data', required=True)
    li.add_argument('--speaker', required=True)
    li.add_argument('--out', required=True)
    a = ap.parse_args(argv)
    return {'transcribe': cmd_transcribe, 'cut': cmd_cut, 'check': cmd_check, 'score': cmd_score, 'list': cmd_list}[a.cmd](a)


if __name__ == '__main__':
    sys.exit(main())

In [ ]:
%%writefile samples_real.py
"""A few Gulf sentences from one exported voice, to listen to.

  python samples_real.py out/ar_AE-saif_gulf-medium.onnx out/samples saif_gulf   (writes out/samples/saif_gulf_1.wav ...)

The text goes through the same steps the site will use for this voice (CATT's diacritics, then the
pausal endings), so the samples sound as the site will.
"""
import sys
import wave
from pathlib import Path

TEXTS = [
    'هلا والله، شحالك اليوم؟ عساك بخير.',
    'بغيت أروح السوق الحين، بس الجو حر وايد.',
    'باجر إن شاء الله بنتلاقى في البيت عقب صلاة المغرب.',
    'شو رايك نطلع البر ونسوي قهوة وشاي على الحطب؟',
    'السموحة منك، ما انتبهت للرسالة إلا الحين.',
]


def main(model, out_dir, name):
    from piper import PiperVoice
    from prep_real import site_vowel
    voice = PiperVoice.load(model)
    vowel = site_vowel()
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    for i, text in enumerate(TEXTS, 1):
        with wave.open(str(out / f'{name}_{i}.wav'), 'wb') as w:
            voice.synthesize_wav(vowel(text), w)
    print('samples:', sorted(p.name for p in out.glob(f'{name}_*.wav')))
    return 0


if __name__ == '__main__':
    sys.exit(main(sys.argv[1], sys.argv[2], sys.argv[3]))

In [ ]:
%%writefile train_main.py
"""`python -m piper.train` with its own checkpointing (same arguments: fit --data... --model...).

piper-tts 1.8.0 keeps the best checkpoints by val_mel and by val_mos, and last.ckpt. Two things went
wrong with that:

- With the MOS predictor off (--model.mos_metric none, or when it cannot be downloaded), val_mos is
  never logged and Lightning stops the whole training at the end of the first epoch
  (MisconfigurationException: could not find the monitored key). Seen in a local dry run.
- Lightning 2.6 writes last.ckpt only in a step where a "best" checkpoint was written. val_mel stops
  improving early in a fine-tune (mel L1 saturates long before the adversarial losses have cleaned
  the sound), so last.ckpt stayed at an early epoch and that early model is what got exported. The
  first real voice (21 Nov) was heard as «صوت هواء ومش واضح»; its last upload said "No files have been
  modified": the checkpoint had not moved in the final half hour of training.

So Piper's checkpoint callbacks go, Lightning's own default one is turned off, and KeepLatest writes
last.ckpt every few epochs and when training ends, whatever the validation numbers say. One file of
~880 MB instead of six.
"""
import os
import sys
from pathlib import Path

from lightning.pytorch.callbacks import Callback

SAVE_EVERY_EPOCHS = 4  # an epoch of 250 clips is ~25 s on a T4; a save is ~880 MB


class KeepLatest(Callback):
    """last.ckpt = the model as it is now. Written under another name and renamed, so whoever copies
    last.ckpt while training runs (the notebook exports samples every half hour) never gets half a file."""

    def __init__(self, every_n_epochs=SAVE_EVERY_EPOCHS):
        self.every = max(1, int(every_n_epochs))

    def _save(self, trainer):
        if trainer.sanity_checking or not trainer.is_global_zero:
            return
        folder = Path(trainer.log_dir or trainer.default_root_dir) / 'checkpoints'
        folder.mkdir(parents=True, exist_ok=True)
        tmp = folder / 'last.ckpt.tmp'
        trainer.save_checkpoint(tmp)
        os.replace(tmp, folder / 'last.ckpt')

    def on_train_epoch_end(self, trainer, pl_module):
        if (trainer.current_epoch + 1) % self.every == 0:
            self._save(trainer)

    def on_train_end(self, trainer, pl_module):
        self._save(trainer)


def main():
    import piper.train.__main__ as piper_train

    piper_train._DEFAULT_CALLBACKS[:] = [KeepLatest()]
    if not any(a.startswith('--trainer.enable_checkpointing') for a in sys.argv):
        sys.argv += ['--trainer.enable_checkpointing', 'false']
    piper_train.main()


if __name__ == '__main__':
    main()

In [ ]:
%%writefile piper_fn.py
"""POST /api/piper  {"text": "...", "voice": "m" | "f"}  ->  audio/wav (22 050 Hz mono)

The two light voices (Piper, formal Arabic: donors v5 and v4, trained in train/) inside this site:
a Python function next to the Node ones, on the same free Vercel project. Hugging Face wants a paid
account for any Gradio or Docker Space now (402 for the setup notebook, public or private), and this
needs no server of its own to set up or keep awake.

Only api/_piper.js calls it, with the header X-Piper-Key: an HMAC of the site's HF_TOKEN, so it needs
no setting of its own and the token itself never travels. The models are read from the private repo
omran77/omran-voices-piper with that token into /tmp (about 60 MB each, once per instance) and kept
while the instance lives. MODEL_DIR (a local folder with the same files) replaces the repo for tests.

The owner gave the first version 6/10: wrong vowels, sentences run together. So the text is vowelled
first by CATT (Abjad AI, Apache-2.0, encoder-only, its released ONNX checked by SHA-256), which got
right what Piper's own diacritizer got wrong (الكتاب not الكتّاب, العِلم not العَلَم, أَشرَح not أُشرِح);
Piper keeps those marks and fills any it left out. And each sentence is followed by a short pause:
Piper writes them back to back. If CATT cannot be fetched, Piper's own diacritics are used.
"""
import hashlib
import hmac
import io
import json
import os
import re
import threading
import time
import wave
import zipfile
from http.server import BaseHTTPRequestHandler

REPO = 'omran77/omran-voices-piper'
FILES = {'m': 'ar_AE-omran_m_fusha-medium.onnx', 'f': 'ar_AE-omran_f_fusha-medium.onnx'}
MAX_CHARS = 1200  # the app sends pieces of ~300 characters; this only stops abuse
MAX_BODY = 16 * 1024
SENTENCE_PAUSE = 0.35  # seconds of silence between sentences
CATT_URL = 'https://github.com/abjadai/catt/releases/download/v2/eo_model_onnx.zip'
CATT_SHA256 = '5343bfdfda70c981b587811b79348b8dde91f65c0a551ca98f232ea5cb62ceac'

voices = {}
lock = threading.Lock()

# CATT's alphabet (Buckwalter letters) and its 18 output tags, as in the model's own tokenizer
AR = dict(zip('ءآأؤإئابةتثجحخدذرزسشصضطظعغفقكلمنهوىي', "'|>&<}AbptvjHxd*rzs$SDTZEgfqklmnhwYy"))
LETTERS = ['<PAD>', '<BOS>', '<EOS>', ' ', '$', '&', "'", '*', '<', '>', 'A', 'D', 'E', 'H', 'S', 'T', 'Y', 'Z',
           'b', 'd', 'f', 'g', 'h', 'j', 'k', 'l', 'm', 'n', 'p', 'q', 'r', 's', 't', 'v', 'w', 'x', 'y', 'z', '|', '}',
           '<MASK>']
TAGS = ['<PAD>', '<BOS>', '<EOS>', '<NT>', '<SD>', '<SDD>', '<SF>', '<SFF>', '<SK>', '<SKK>',
        'F', 'K', 'N', 'a', 'i', 'o', 'u', '~']
MARKS = {'F': 'ً', 'N': 'ٌ', 'K': 'ٍ', 'a': 'َ', 'u': 'ُ', 'i': 'ِ', '~': 'ّ',
         'o': 'ْ', '<SD>': 'ُّ', '<SDD>': 'ٌّ', '<SF>': 'َّ',
         '<SFF>': 'ًّ', '<SK>': 'ِّ', '<SKK>': 'ٍّ'}
LETTER_ID = {c: i for i, c in enumerate(LETTERS)}
HARAKAT = re.compile('[ً-ْـٰ]')
NOT_ARABIC = re.compile('[^' + ''.join(AR) + ']')


class Catt:
    def __init__(self, folder):
        import onnxruntime as ort
        self.enc = ort.InferenceSession(os.path.join(folder, 'encoder.onnx'), providers=['CPUExecutionProvider'])
        self.dec = ort.InferenceSession(os.path.join(folder, 'decoder.onnx'), providers=['CPUExecutionProvider'])

    def tags(self, words):
        import numpy as np
        ids = np.array([[LETTER_ID[' '] if c == ' ' else LETTER_ID[AR[c]] for c in words]], dtype=np.int64)
        mask = np.ones((1, 1, ids.shape[1], ids.shape[1]), dtype=bool)
        logits = self.dec.run(None, {'enc_src': self.enc.run(None, {'src': ids, 'src_mask': mask})[0]})[0]
        return [TAGS[i] for i in logits[0].argmax(-1)]

    def sentence(self, text):
        # the model reads the Arabic words only; its marks go back on the same letters, everything else stays
        bare = HARAKAT.sub('', text)
        words = ' '.join(NOT_ARABIC.sub(' ', bare).split())
        if not words:
            return bare
        marks = iter([t for c, t in zip(words, self.tags(words)) if c != ' '])
        return ''.join(c + MARKS.get(next(marks, ''), '') if c in AR else c for c in bare)

    def __call__(self, text):  # a sentence at a time, as it was trained
        return ''.join(self.sentence(p) for p in re.split(r'([.!?؟]+)', text))


catt_state = {'model': None, 'failed_at': 0.0}


def catt():
    # the released model, fetched once per instance; after a failure, tried again ten minutes later
    with lock:
        if catt_state['model'] or time.time() - catt_state['failed_at'] < 600:
            return catt_state['model']
        try:
            folder = os.environ.get('CATT_DIR') or '/tmp/catt'
            if not os.path.exists(os.path.join(folder, 'encoder.onnx')):
                import urllib.request
                with urllib.request.urlopen(CATT_URL, timeout=60) as r:
                    data = r.read()
                if hashlib.sha256(data).hexdigest() != CATT_SHA256:
                    raise ValueError('the CATT download does not match its checksum')
                zipfile.ZipFile(io.BytesIO(data)).extractall(folder, members=['encoder.onnx', 'decoder.onnx'])
            catt_state['model'] = Catt(folder)
        except Exception as e:  # Piper's own diacritics still work
            print('catt failed:', type(e).__name__, str(e)[:300])
            catt_state['failed_at'] = time.time()
        return catt_state['model']


def hf_token():
    # same rule as hfToken() in api/_engine.js: invisible marks out, and only a real-looking token
    t = re.sub(r'[\s​-‏‪-‮⁦-⁩﻿]', '', os.environ.get('HF_TOKEN', ''))
    return t if re.fullmatch(r'hf_[A-Za-z0-9]{20,}', t) else ''


def site_key(token):
    return hmac.new(token.encode(), b'omran-voices-piper', hashlib.sha256).hexdigest() if token else ''


def fetch(name, token):
    local = os.environ.get('MODEL_DIR')
    if local:
        return os.path.join(local, name)
    os.environ.setdefault('HF_HOME', '/tmp/hf')  # the only place a function may write
    from huggingface_hub import hf_hub_download
    return hf_hub_download(REPO, name, token=token)


def voice_for(spk, token):
    with lock:  # one download and load per voice, even when two requests arrive together
        if spk not in voices:
            from piper import PiperVoice
            model = fetch(FILES[spk], token)
            fetch(FILES[spk] + '.json', token)
            voices[spk] = PiperVoice.load(model)  # Arabic: Piper adds any diacritics still missing (as in training)
        return voices[spk]


def speak(voice, text):
    vowel = catt()
    if vowel:
        text = vowel(text)
    buf = io.BytesIO()
    with wave.open(buf, 'wb') as w:
        w.setframerate(voice.config.sample_rate)
        w.setsampwidth(2)
        w.setnchannels(1)
        for i, chunk in enumerate(voice.synthesize(text)):  # one chunk per sentence
            if i:
                w.writeframes(bytes(2 * int(voice.config.sample_rate * SENTENCE_PAUSE)))
            w.writeframes(chunk.audio_int16_bytes)
    return buf.getvalue()


class handler(BaseHTTPRequestHandler):
    def reply(self, code, body, kind='application/json'):
        if kind == 'application/json':
            body = json.dumps(body, ensure_ascii=False).encode()
        self.send_response(code)
        self.send_header('Content-Type', kind)
        self.send_header('Cache-Control', 'no-store')
        self.send_header('Content-Length', str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_POST(self):
        token = hf_token()
        key = site_key(token)
        if not key or not hmac.compare_digest(self.headers.get('X-Piper-Key', '').encode(), key.encode()):
            return self.reply(401, {'error': 'key'})
        size = int(self.headers.get('Content-Length') or 0)
        if not 0 < size <= MAX_BODY:
            return self.reply(400, {'error': 'body'})
        try:
            req = json.loads(self.rfile.read(size))
            text, spk = ' '.join(str(req.get('text', '')).split()), str(req.get('voice', ''))
        except (ValueError, AttributeError):
            return self.reply(400, {'error': 'json'})
        if not text or len(text) > MAX_CHARS:
            return self.reply(400, {'error': 'text'})
        if spk not in FILES:
            return self.reply(404, {'error': 'voice'})
        try:
            voice = voice_for(spk, token)
        except Exception as e:  # the repo, the token's access, or the download: in the owner's Vercel logs
            print('piper model failed:', type(e).__name__, str(e).replace(token, '***')[:300])
            return self.reply(502, {'error': 'model'})
        return self.reply(200, speak(voice, text), 'audio/wav')

    def do_GET(self):
        self.reply(405, {'error': 'POST'})

In [ ]:
# Two separate Python 3.11 environments: Whisper (writes the recordings as text) and Piper (trains),
# away from the notebook's own packages. A file ".ok" marks an environment as finished.
run('pip install -q uv')
PY_W, PY_B = str(W / 'envW/bin/python'), str(W / 'envB/bin/python')
TORCH = 'torch==2.6.0 torchaudio==2.6.0'
CU = '--index-url https://download.pytorch.org/whl/cu124'
if not Path('envW/.ok').exists():
    run('rm -rf envW && uv venv envW --python 3.11 --seed -q')
    run(f'uv pip install -q --python {PY_W} {TORCH} {CU}')
    run(f'uv pip install -q --python {PY_W} openai-whisper numpy scipy {TORCH}')
    Path('envW/.ok').touch()
if not Path('envB/.ok').exists():
    # Piper from PyPI (its espeak part comes built). Building it from source needs cmake, and
    # cmake 4 refuses older projects.
    run('rm -rf envB && uv venv envB --python 3.11 --seed -q')
    run(f'uv pip install -q --python {PY_B} {TORCH} {CU}')
    # Lightning is pinned: 2.6 changed when last.ckpt is written (train_main.py), and the next change should not find us on a 10-hour run.
    run(f'uv pip install -q --python {PY_B} "piper-tts[train]==1.8.0" "lightning==2.6.6" cython numpy "setuptools<81" huggingface_hub {TORCH}')
    # The alignment step is a small Cython file the PyPI package leaves out: build it from Piper's source.
    if not Path('piper1-gpl').exists():
        run('git clone -q --depth 1 --branch v1.8.0 https://github.com/OHF-voice/piper1-gpl.git piper1-gpl'
            ' || git clone -q --depth 1 https://github.com/OHF-voice/piper1-gpl.git piper1-gpl')
    mab = Path('mabuild')
    mab.mkdir(exist_ok=True)
    shutil.copy('piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx', mab / 'core.pyx')
    (mab / 'setup.py').write_text('from setuptools import setup, Extension\n'
                                  'from Cython.Build import cythonize\n'
                                  'import numpy\n'
                                  "setup(ext_modules=cythonize([Extension('core', ['core.pyx'], include_dirs=[numpy.get_include()])]))\n")
    run(f'cd mabuild && {PY_B} setup.py -q build_ext --inplace')
    dest = Path(out([PY_B, '-c', 'import piper, os; print(os.path.dirname(piper.__file__))'])) / 'train/vits/monotonic_align/monotonic_align'
    dest.mkdir(parents=True, exist_ok=True)
    (dest / '__init__.py').touch()
    for so in mab.glob('core*.so'):
        shutil.copy(so, dest)
    Path('envB/.ok').touch()
run([PY_W, '-c', 'import whisper, torch, scipy; print("whisper ok, GPU:", torch.cuda.is_available())'])
run([PY_B, '-c', 'from piper.train.vits.monotonic_align import maximum_path; from piper.phonemize_espeak import EspeakPhonemizer;'
                 ' print("piper ok", EspeakPhonemizer().phonemize("ar", "هلا والله"))'])

In [ ]:
# The person's recordings (private dataset) and our sentences.
key = SPEAKER.strip().lower().replace(' ', '_')
folders = [f.path for f in api.list_repo_tree(REC_REPO, path_in_repo='data', repo_type='dataset')
           if f.path.split('/')[-1].rsplit('-', 1)[0] == key]
assert folders, f'ما لقيت تسجيلات باسم «{SPEAKER}» في {REC_REPO}'
print('recordings of', SPEAKER, ':', folders)
snapshot_download(REC_REPO, repo_type='dataset', allow_patterns=[f + '/**' for f in folders], local_dir='rec', token=HF_TOKEN)
REC = 'rec/data'

sentences = []
for url in SENTENCE_URLS:
    try:
        with urllib.request.urlopen(url, timeout=60) as r:
            sentences += json.loads(r.read().decode('utf-8'))
    except Exception as e:  # the dialect file may not be there: the 10 000 are enough
        print('skipped', url, type(e).__name__)
assert sentences, 'ما قدرت أحمّل الجمل'
Path('sentences.json').write_text(json.dumps(sentences, ensure_ascii=False), encoding='utf-8')
print(len(sentences), 'sentences')

In [ ]:
# Step 1: the recordings become clips with their text. Whisper's words are kept in the repo, so a
# second run only transcribes what was recorded since.
WORDS = f'data/real-{VOICE}-words.json'
if WORDS in REPO_FILES and not Path('words.json').exists():
    shutil.copy(hf_hub_download(MODEL_REPO, WORDS, token=HF_TOKEN), 'words.json')
gpu_env = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
run([PY_W, 'prep_real.py', 'transcribe', '--rec', REC, '--sentences', 'sentences.json', '--out', 'words.json', '--model', WHISPER], env=gpu_env)
api.upload_file(path_or_fileobj='words.json', path_in_repo=WORDS, repo_id=MODEL_REPO, commit_message='words ' + VOICE)

shutil.rmtree('data', ignore_errors=True)
run([PY_W, 'prep_real.py', 'cut', '--rec', REC, '--words', 'words.json', '--sentences', 'sentences.json', '--speaker', VOICE, '--out', 'data'])
run([PY_W, 'prep_real.py', 'check', '--data', 'data', '--speaker', VOICE, '--model', WHISPER], env=gpu_env)
report = json.loads(Path(f'data/{VOICE}.report.json').read_text(encoding='utf-8'))
print(f"{report['clips']} clips, {report['seconds'] / 60:.1f} minutes, {report['matched']} with the screen's spelling, misheard and dropped: {report.get('misheard', 0)}")
sound = report.get('sound', {})
print('SOUND: the recording is', sound.get('muffled_db'), 'dB under clear speech at 1.5-4 kHz;',
      f"repaired, now {sound.get('after_db')} dB" if sound.get('repaired') else 'clear enough, left as it is')
assert report['seconds'] >= 5 * 60, 'أقل من ٥ دقايق كلام صالح للتدريب: سجّل أكثر'
run([PY_B, 'prep_real.py', 'list', '--data', 'data', '--speaker', VOICE, '--out', f'data/{VOICE}.csv'])
with tarfile.open('real-data.tar', 'w') as t:
    t.add('data')
api.upload_file(path_or_fileobj='real-data.tar', path_in_repo=f'data/real-{VOICE}.tar', repo_id=MODEL_REPO, commit_message='training data ' + VOICE)

In [ ]:
# Step 2: the voice starts from the Arabic voice "kareem" with every layer in place (or continues
# from its own last saved version), and moves toward the person's voice and way of speaking.
NAME = VOICE_FILE.format(VOICE)
CKPT = f'ckpt/{VOICE}_gulf_r{ROUND}.ckpt'
left = min(TRAIN_HOURS, hours_left() - 0.6)  # export and upload need ~30 minutes
assert left > 0.3, 'not enough time left in this session'
hh, mm = int(left), int((left - int(left)) * 60)
Path('out').mkdir(exist_ok=True)
if CKPT in REPO_FILES:
    warm = hf_hub_download(MODEL_REPO, CKPT, token=HF_TOKEN)
    print('continues from its last saved version')
else:
    kareem = [f.path for f in HfApi().list_repo_tree('rhasspy/piper-checkpoints', path_in_repo='ar/ar_JO/kareem/medium',
                                                     repo_type='dataset') if f.path.endswith('.ckpt')][0]
    warm = hf_hub_download('rhasspy/piper-checkpoints', kareem, repo_type='dataset')
    print('starts from', kareem)
cmd = [PY_B, 'train_main.py', 'fit',
       '--data.voice_name', NAME, '--data.csv_path', f'data/{VOICE}.csv', '--data.audio_dir', 'data/wav',
       '--model.sample_rate', '22050', '--data.espeak_voice', 'ar', '--data.cache_dir', f'cache/{VOICE}',
       '--data.trim_silence', 'false',  # prep_real.py cut the clips; Piper's own trim can leave a clip too short to train on
       '--data.config_path', f'out/{NAME}.onnx.json', '--data.batch_size', '16', '--data.num_workers', '2',
       '--model.warmstart_ckpt', warm, '--model.mos_metric', 'none',
       '--trainer.accelerator', 'gpu', '--trainer.devices', '1', '--trainer.max_time', f'00:{hh:02d}:{mm:02d}:00',
       '--trainer.default_root_dir', f'runs/{VOICE}']
print(' '.join(cmd))
train = subprocess.Popen(cmd, env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), stdout=open('train.log', 'w'), stderr=subprocess.STDOUT)


def export():
    """The newest checkpoint -> the voice file and its samples. Returns where in training it is from, or ''."""
    cks = sorted(W.glob(f'runs/{VOICE}/lightning_logs/version_*/checkpoints/last.ckpt'), key=lambda p: p.stat().st_mtime)
    if not cks:
        print('no checkpoint yet')
        return ''
    shutil.copy(cks[-1], 'snap.ckpt')
    try:
        at = out([PY_B, '-c', "import torch; c = torch.load('snap.ckpt', map_location='cpu', weights_only=False); "
                              "print('epoch', c['epoch'], 'step', c['global_step'])"])
        run([PY_B, '-m', 'piper.train.export_onnx', '--checkpoint', 'snap.ckpt', '--output-file', f'out/{NAME}.onnx'], log=False)
        run([PY_B, 'samples_real.py', f'out/{NAME}.onnx', 'out/samples', f'{VOICE}_gulf'], log=False)
    except (RuntimeError, subprocess.CalledProcessError) as e:
        print('export failed this time:', e)
        return ''
    return at


SCORES = []  # (time, where in training, clarity)


def clarity():
    """How much of the sample sentences Whisper hears in the samples, 0-1, on the card that is not
    training (the voice the training starts from is ~0.97; the first run's «مش واضح» was 0.76)."""
    try:
        env = dict(os.environ, CUDA_VISIBLE_DEVICES='1' if len(gpus) > 1 else '')
        text = out([PY_W, 'prep_real.py', 'score', '--samples', 'out/samples', '--name', f'{VOICE}_gulf', '--model', WHISPER], env=env)
        return float(text.split('SCORE')[-1].split()[0])
    except Exception as e:  # a measurement must never stop the training
        print('clarity not measured this time:', type(e).__name__, str(e)[-200:])
        return None


def upload(final=False):
    ok = export()
    score = clarity() if ok else None
    if ok:
        SCORES.append((time.strftime('%H:%M'), ok, score))
        if score is not None and score >= max([s for _, _, s in SCORES[:-1] if s is not None], default=-1.0):
            # the clearest version so far is kept beside the newest: nine hours on twenty minutes of
            # speech can pass its best point
            shutil.rmtree('out/best', ignore_errors=True)
            Path('out/best').mkdir(parents=True)
            for f in [Path(f'out/{NAME}.onnx'), Path(f'out/{NAME}.onnx.json')] + sorted(Path('out/samples').glob(f'{VOICE}_gulf_*.wav')):
                shutil.copy(f, 'out/best')
            Path('out/best/from.txt').write_text(f'{ok}, clarity {score}\n')
    Path('out/logs').mkdir(parents=True, exist_ok=True)
    for f in W.glob('*.log'):
        shutil.copy(f, 'out/logs')
    shutil.copy(f'data/{VOICE}.report.json', 'out/logs')
    api.upload_folder(folder_path='out', repo_id=MODEL_REPO, commit_message=f'{VOICE} voice' + (' (final)' if final else ''))
    if ok:
        api.upload_file(path_or_fileobj='snap.ckpt', path_in_repo=CKPT, repo_id=MODEL_REPO, commit_message='checkpoint ' + VOICE)
    print(time.strftime('%H:%M'), 'uploaded to', MODEL_REPO, f'(model and samples from {ok}, clarity {score})' if ok else '(logs only)')
    return ok


last_up = time.time()
while train.poll() is None:
    time.sleep(300)
    lines = Path('train.log').read_text(errors='replace').splitlines()
    print(time.strftime('%H:%M'), (lines[-1] if lines else '...')[-200:])
    if train.poll() is None and time.time() - last_up > UPLOAD_EVERY_MIN * 60:
        upload()
        last_up = time.time()
print('training ended, code', train.returncode)
print(Path('train.log').read_text(errors='replace')[-3000:])
DONE = upload(final=True)
print('CLARITY through the training (1 = every letter of the sample sentences heard; the starting voice ~0.97, the first run 0.76):')
for t, at, s in SCORES:
    print(' ', t, at, s)
print('done in', round((time.time() - T0) / 3600, 1), 'hours')

In [ ]:
# اسمع العيّنات هني. نفسها محفوظة في المستودع الخاص تحت samples/
from IPython.display import Audio, display
assert DONE, 'التدريب ما طلّع نموذج: شوف السجل فوق'
print('آخر نسخة من التدريب:', DONE)
for p in sorted(Path('out/samples').glob(f'{VOICE}_gulf_*.wav')):
    print(p.name)
    display(Audio(filename=str(p)))
best = Path('out/best/from.txt')
if best.exists() and not best.read_text().startswith(DONE + ','):
    print('أوضح نسخة أثناء التدريب (بقياس الآلة):', best.read_text().strip())
    for p in sorted(Path('out/best').glob(f'{VOICE}_gulf_*.wav')):
        print(p.name)
        display(Audio(filename=str(p)))